# MSCAD multi-stage intrusion-detection pipeline

Step 1 (PCAPNG -> one flow CSV per capture) lives in
`convert_pcapng_to_csvflows.ipynb`. This notebook runs **Step 2** and **Step 3**,
which are now separate stages.

| Step | What it does | Reads | Writes |
|------|--------------|-------|--------|
| **2 - Data acquisition** | one DataFrame per capture, tagged and reported | `C:\MSCAD2022\CSVs` | `C:\MSCAD2022\outputs\step2` |
| **3 - Dataset merging** | concatenate, verify, save, inspect | the Step 2 frames | `C:\MSCAD2022\outputs\step3` |
| 4+ | preprocessing onward | the merged dataset | later |

**Every path is absolute and lives on the data drive.** Nothing is resolved
against the notebook's working directory, so the outputs land in the same place
no matter where Jupyter was started or where this code is checked out.

## Setup

In [ ]:
import sys
from pathlib import Path

# The notebooks live in PROJECT_ROOT\notebooks, so when Jupyter is started in
# that folder the project root is one level up.  Starting Jupyter in the project
# root instead also works.  Nothing is hard-coded, so this is correct whether the
# code sits in C:\MSCAD2022\Notebook or in a git clone.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

%load_ext autoreload
%autoreload 2

import config

print("project root :", PROJECT_ROOT)
print("kernel python:", sys.version.split()[0])
print("flow CSVs in :", config.CSV_OUTPUT_DIR)
print("merged out   :", config.MERGED_DIR)
print("step 2 out   :", config.STEP2_OUTPUT_DIR)
print("kaggle ref   :", config.KAGGLE_CSV_DIR,
      "(exists)" if config.KAGGLE_CSV_DIR.is_dir() else "(not present - section 13 is skipped)")
print("random seed  :", config.RANDOM_SEED)

---
# Step 2 - Data acquisition

Loads every `<capture>_flows.csv` from `config.CSV_INPUT_DIR`
(`C:\MSCAD2022\CSVs`) into one DataFrame per capture, each tagged with:

| Column | Meaning |
|---|---|
| `Source_Capture` | the file stem the row came from |
| `Orig_Row` | the row's position inside its own file |

Nothing is merged, cleaned or reordered here. If the folder is missing or holds
no matching CSVs, this **stops** with a message naming the folder. If a single
file fails to read, it is reported and the rest still load.

### 2.1 Load the per-capture CSVs

In [ ]:
from src.mscad import data_loading

frames, acquisition_report, failures = data_loading.acquire_captures()

print()
print("captures loaded:", list(frames))
print("failures       :", failures if failures else "none")

### 2.2 What each capture contains

One row per file: name, rows, columns, size, label counts and the first/last
timestamp. Saved to `C:\MSCAD2022\outputs\step2\acquisition_report.csv`.

In [ ]:
data_loading.show_table(acquisition_report)

---
# Step 3 - Dataset merging

Concatenates the Step 2 frames in `config.CAPTURE_ORDER`, sorts by
`Timestamp_dt` with a stable mergesort, saves to
`C:\MSCAD2022\outputs\step3\MSCAD_merged.csv` and then **verifies the file on
disk**. Values are never changed: `inf`, `NaN` and duplicate rows all survive.

### 3.1 Column consistency

Every capture must carry the same columns in the same order. A mismatch is
**reported in a table**, never silently filled -- `pd.concat` would quietly
insert `NaN` that looks like missing data later instead of like the bug it is.

In [ ]:
consistency, columns_ok = data_loading.check_column_consistency(frames)

assert columns_ok, "the captures do not share one column set -- see the table above"

### 3.2 Merge

In [ ]:
merged, _ = data_loading.merge_flow_csvs(frames=frames)

### 3.3 Merge checks - PASS/FAIL with numbers

Row totals, per-capture counts, every `Label` count, the column layout, and the
`NaN` / `inf` totals, each against the per-file numbers from Step 2.

In [ ]:
infos = [{"capture": c,
          "rows": len(f),
          "label_counts": f[config.LABEL_COLUMN].value_counts().to_dict(),
          "nan_cells": int(f[[c2 for c2 in config.KAGGLE_FEATURES if c2 in f.columns]]
                           .isna().to_numpy().sum()),
          "inf_cells": int(__import__("numpy").isinf(
              f[[c2 for c2 in config.KAGGLE_FEATURES if c2 in f.columns]]
              .to_numpy(dtype="float64")).sum()),
          "timestamp_format": None,
          "first_timestamp": f["Timestamp_dt"].min(),
          "last_timestamp": f["Timestamp_dt"].max()}
         for c, f in frames.items()]

checks, all_passed = data_loading.check_merge(merged, infos)

print()
print("ALL MERGE CHECKS PASSED:", all_passed)
assert all_passed, "the merge is not faithful -- see the FAIL rows above"

### 3.4 Save the merged dataset, and prove it is on disk

Writes `C:\MSCAD2022\outputs\step3\MSCAD_merged.csv`, creating the folder if
needed, then re-reads it and compares the row and column counts with the
in-memory frame. You get `[OK]` or a clear `[FAIL]` and the absolute path.

If the file already exists and `config.OVERWRITE_MERGED` is `False`, it is
**not** rewritten -- that is said plainly and the existing file is loaded and
verified instead.

In [ ]:
save_result = data_loading.save_merged(merged, checks=checks, infos=infos)

print()
print("status :", save_result["status"])
print("path   :", save_result["target"])
print("ok     :", save_result["ok"])
assert save_result["ok"], "the merged file is not on disk, or does not match"

### 3.5 Load the merged dataset back from disk

In [ ]:
dataset = data_loading.load_merged()

print("shape from disk:", dataset.shape)
print("matches the in-memory merge:", dataset.shape == merged.shape)

### 3.6 Step 3 report files

`label_counts.csv`, `source_capture_x_label.csv`, `label_counts.png` and
`summary.txt`, all in `C:\MSCAD2022\outputs\step3`.

In [ ]:
reports = data_loading.save_step3_reports(dataset)

### 3.7 Inspect the merged dataset

Shape and memory, `head(10)` / `tail(5)` with all columns, dtypes and null
counts, label counts, the `Source_Capture` x `Label` crosstab, per-capture
timestamps, protocols, `NaN` / `inf` per column, duplicate counts, `describe()`
with an `inf_count` column, the label chart, and the Kaggle comparison.

Everything **counts and describes** - nothing is dropped, filled or changed.

In [ ]:
summary = data_loading.summarize_dataset(dataset)

---
## What Step 4 has to deal with

1. **`inf` in the rate columns** for zero-duration flows -- correct arithmetic,
   but sklearn and Keras both reject it. Clip, impute or drop: an explicit choice.
2. **`NaN`** wherever section 9 is non-zero -- per-column decisions, not a
   blanket `fillna(0)`.
3. **Identical-feature rows** -- real observations in floods, scans and brute
   force, but a random split puts identical rows on both sides and inflates
   every metric. Group-aware splitting is the defence.
4. **Class imbalance** -- see the ratio in section 5. Accuracy will be useless;
   macro-F1 and per-class recall are the metrics to watch.
5. **The identifier columns are label leakage.** Attacker IPs decided the labels
   in Step 1, so `Src IP` / `Dst IP` must not be features -- but they must
   survive for the sequence mining and attack-state graph.

---
# Stage 2: Load & merge flow CSVs

**What this stage does**

Reads the six per-capture CSVs Step 1 wrote (`<stem>_flows.csv`, 73 columns:
6 flow identifiers + 66 MSCAD features + `Label`), checks each header, and
concatenates them into one dataset with three bookkeeping columns added:

| Added column | Meaning |
|---|---|
| `Source_Capture` | which capture the row came from, taken from the file name |
| `Orig_Row` | the row's position inside its own file, so any row can be traced back |
| `Timestamp_dt` | the `Timestamp` text parsed to a real datetime; the text column is kept |

76 columns in total. Rows are then put in **time order** with a stable sort, so
the merged file reads like the attack session it came from — which is what the
Stage 8 sequence mining and attack-state graph need.

**What this stage deliberately does NOT do**

No cleaning: `inf`, `-inf`, `NaN` and duplicate rows are all carried through
exactly as they are, and the class imbalance is left alone. No encoding, no
scaling, no train/test split, no feature selection, no windowing, no training.
Those are Stage 3 onward. Step 2's only job is to produce one faithful merged
dataset and describe it honestly.

---
# Step 4 - Preprocessing

Cleans, encodes and scales the **whole** merged dataset from Step 3.

**In scope:** inf and NaN handling, dropping zero-variance columns, separating
the non-feature columns, label encoding, a binary target, Protocol one-hot,
and scaling.

**Deliberately NOT in scope:** no train/test split, no class-imbalance handling
(no SMOTE, no resampling, no class weights), no feature selection, no
modelling. Those are later steps.

| | |
|---|---|
| Input | `C:\MSCAD2022\outputs\step3\MSCAD_merged.csv` |
| Output | `C:\MSCAD2022\outputs\step4\` |
| Settings | `INF_STRATEGY`, `SCALER`, `LABEL_MAP`, `OVERWRITE_STEP4` in `config.py` |

Duplicate rows are **counted, never dropped** - the sequence work in a later
step depends on the original row order, and row *i* of `features`, `labels` and
`metadata` always refers to the same flow.

### Step 4 settings

In [ ]:
from src.mscad import preprocessing

print("input      :", config.MERGED_CSV)
print("output     :", config.STEP4_OUTPUT_DIR)
print("INF_STRATEGY   :", config.INF_STRATEGY, "  ('cap' or 'nan_median')")
print("SCALER         :", config.SCALER, "  ('minmax' or 'standard')")
print("LABEL_MAP      :", config.LABEL_MAP or "{} (nothing renamed)")
print("BENIGN_LABELS  :", config.BENIGN_LABELS, "-> Label_binary 0")
print("OVERWRITE_STEP4:", config.OVERWRITE_STEP4)
print("metadata kept aside:", len(config.METADATA_COLUMNS), "columns")

## 4.1 Clean

Runs in this order, each reported in its own table:

| | |
|---|---|
| **a** | `inf` -> capped at the column's largest finite value, plus a `zero_duration` marker |
| **b** | `NaN` -> median of its own column (never a blanket zero) |
| **c** | constant / zero-variance columns dropped, **measured not assumed** |
| **d** | non-feature columns moved to `metadata`, keeping row order |
| **e** | duplicate feature rows **counted, not dropped** |
| **f** | exact label strings reported; `LABEL_MAP` applied only if set |

## 4.2 Encode

`Label` -> `Label_encoded` (LabelEncoder, mapping saved as JSON) and
`Label_binary` (0 for `Benign`/`Normal`, 1 otherwise - they stay separate
classes in the multi-class target). `Protocol` -> four one-hot columns, with
anything unexpected landing in `Protocol_other`.

## 4.3 Scale

`SCALER` over the numeric features only - one-hot and binary columns keep their
0/1 meaning. **Both the scaled and the unscaled cleaned features are saved**:
this scaler is fitted on the whole dataset, which is fine for inspection but
would leak test information once a split exists, so a later step can refit it
on the training split alone from `features_clean.csv`.

### Run Step 4

In [ ]:
step4 = preprocessing.run_step4()

### What came out

`features_clean.csv`, `features_scaled.csv`, `labels.csv`, `metadata.csv`,
`label_mapping.json`, `feature_list.json`, `preprocess_artifacts.joblib` and
`step4_report.csv`, all in `C:\MSCAD2022\outputs\step4`. Every CSV was
re-read from disk and its row count compared with the frame in memory.

In [ ]:
if step4.get("status") == "loaded_existing":
    print("Existing Step 4 outputs were loaded (config.OVERWRITE_STEP4 is False).")
    print("Set OVERWRITE_STEP4 = True to recompute them.")
else:
    preprocessing.show_table(step4["report"])
    print()
    print("all checks passed:", step4["ok"])

### Row alignment

Row *i* of the features, labels and metadata is the same flow. This is what the
Step 8 sequence mining and attack-state graph depend on.

In [ ]:
clean, labels, meta = (step4["features_clean"], step4["labels"], step4["metadata"]) \
    if "features_clean" in step4 else (None, None, None)

if clean is not None:
    print("features:", clean.shape, "| labels:", labels.shape, "| metadata:", meta.shape)
    preprocessing.show_table(
        meta.head(5).join(labels.head(5)[["Label", "Label_encoded", "Label_binary"]]))

## 4.4 Final preprocessed dataset (`preprocessing.csv`)

One file laid out for **reading** and for the sequence / attack-chain work, in
the original row order:

```
Src IP | Dst IP | Src Port | Dst Port | Protocol | Timestamp | <scaled features> | Label
```

- The six identifiers carry their **original** values from the Step 3 merged
  data: IP strings unchanged, ports as integers, `Timestamp` as the original
  text. None of them is scaled.
- `Protocol` is **one readable column** - `TCP`, `UDP`, `ICMP` or `OTHER` -
  derived from the four `Protocol_*` one-hot columns. Exactly one of the four is
  `1` in every row, so the mapping is lossless and reversible, and a row that
  does not have exactly one set is reported as `[FAIL]` rather than guessed at.
  The four one-hot columns themselves are **not** in this file.
- The last column is the original `Label` string. `Label_encoded` and
  `Label_binary` are **not** here; `labels.csv` still has all three.

> **This file is the readable view, not the training matrix.** Two reasons:
> the identifier columns leak the answer - the attacker IPs decided the labels
> back in Step 1, so a model trained on them would score near-perfectly while
> learning nothing - and `Protocol` is text here, which a model cannot consume.
>
> For model training use `features_scaled.csv` (or `features_clean.csv`), which
> keep the numeric `Protocol_*` one-hot columns and exclude the identifiers
> entirely, and `labels.csv` for the encoded targets:
>
> ```python
> features = pd.read_csv(config.STEP4_OUTPUT_DIR / "features_scaled.csv")
> labels = pd.read_csv(config.STEP4_OUTPUT_DIR / "labels.csv")
> X = preprocessing.get_feature_matrix(features)   # identifiers and labels excluded
> y = labels["Label_encoded"]
> ```
>
> `config.ID_COLUMNS_NOT_FOR_TRAINING` names the columns that must never be fed
> to a model, and they are not in `feature_list.json`.

Set `config.COMBINED_ID_COLUMNS = []` for the features-and-labels-only layout,
or `COMBINED_USE_SCALED = False` to carry the unscaled features instead.


In [ ]:
import pandas as pd

combined_path = config.COMBINED_CSV
if not config.usable_output_dir(config.STEP4_OUTPUT_DIR) or not combined_path.exists():
    print("preprocessing.csv not found at", combined_path)
    print("Run the Step 4 cell above first.")
    final = None
else:
    final = pd.read_csv(combined_path, low_memory=False)
    size_mb = combined_path.stat().st_size / 1024 / 1024
    preprocessing.show_table(preprocessing.kv_table([
        ("absolute path", str(combined_path)),
        ("size (MB)", round(size_mb, 2)),
        ("rows", f"{final.shape[0]:,}"),
        ("columns", final.shape[1]),
        ("memory, deep (MB)", round(final.memory_usage(deep=True).sum() / 1024 / 1024, 1)),
    ]))

### Which columns are identifiers, features and labels

In [ ]:
if final is not None:
    ids = [c for c in config.COMBINED_ID_COLUMNS if c in final.columns]
    labels_cols = [c for c in config.COMBINED_LABEL_COLUMNS if c in final.columns]
    feature_cols = [c for c in final.columns if c not in ids + labels_cols]

    preprocessing.show_table(pd.DataFrame([
        {"role": "identifier (NOT a feature)", "columns": len(ids),
         "names": ", ".join(ids)},
        {"role": "feature (model input)", "columns": len(feature_cols),
         "names": ", ".join(feature_cols[:6]) + (", ..." if len(feature_cols) > 6 else "")},
        {"role": "label (target)", "columns": len(labels_cols),
         "names": ", ".join(labels_cols)},
        {"role": "TOTAL", "columns": final.shape[1], "names": ""},
    ]), note="`Protocol` is an identifier here, as readable text. The four numeric "
             "`Protocol_*` one-hot columns a model trains on are in "
             "`features_scaled.csv`, not in this file.")

    left_out = [c for c in config.COMBINED_DROP_FEATURE_COLUMNS if c in final.columns]
    extra = [c for c in ("Label_encoded", "Label_binary") if c in final.columns]
    print("[OK]   no Protocol_* one-hot columns" if not left_out
          else "[FAIL] found %s" % left_out)
    print("[OK]   no Label_encoded / Label_binary columns" if not extra
          else "[FAIL] found %s" % extra)

### head(10) and tail(5), all columns

In [ ]:
if final is not None:
    preprocessing.show_table(final.head(10), all_columns=True,
                             note="All %d columns; scroll sideways." % final.shape[1])
    preprocessing.show_table(final.tail(5), all_columns=True)

### Protocol counts

`Protocol` is the readable form of the four one-hot columns. The counts below
must match the one-hot sums in `features_scaled.csv` exactly - that is what the
verification table checks.


In [ ]:
if final is not None:
    counts = final["Protocol"].value_counts()
    rows = []
    for name in config.PROTOCOL_TEXT_VALUES:
        rows.append({"Protocol": name, "count": int(counts.get(name, 0)),
                     "%": round(100.0 * int(counts.get(name, 0)) / len(final), 3)})
    rows.append({"Protocol": "TOTAL", "count": len(final), "%": 100.0})
    preprocessing.show_table(pd.DataFrame(rows))

    unexpected = sorted(set(final["Protocol"].dropna().astype(str))
                        - set(config.PROTOCOL_TEXT_VALUES))
    blank = int(final["Protocol"].isna().sum())
    print("[OK]   Protocol holds only %s" % "/".join(config.PROTOCOL_TEXT_VALUES)
          if not unexpected else "[FAIL] unexpected values: %s" % unexpected)
    print("[OK]   no blank Protocol cells" if blank == 0
          else "[FAIL] %d blank Protocol cells" % blank)

### Label counts in the combined file

In [ ]:
if final is not None:
    counts = final["Label"].value_counts()
    table = pd.DataFrame({"Label": counts.index.astype(str),
                          "count": counts.to_numpy(),
                          "%": (100.0 * counts / len(final)).round(3).to_numpy()})
    table = pd.concat([table, pd.DataFrame([{"Label": "TOTAL", "count": len(final),
                                             "%": 100.0}])], ignore_index=True)
    preprocessing.show_table(table)

### Verification

Re-read from disk and checked against the other Step 4 files: the column order,
the absence of the `Protocol_*` and `Label_encoded`/`Label_binary` columns, the
identifier values against `metadata.csv` row by row, the `Protocol` counts
against the one-hot sums, the labels against `labels.csv`, and the feature values
against `features_scaled.csv`.


In [ ]:
if final is not None and "step4" in dir() and "features_scaled" in step4:
    table, ok, _ = preprocessing.verify_combined(
        final, step4["features_scaled"], step4["labels"], step4["metadata"],
        expected_rows=len(step4["features_scaled"]))
    print()
    print("ALL CHECKS PASSED:", ok)
elif final is not None:
    print("Step 4 was loaded from disk, so the in-memory frames are not available.")
    print("Re-run the Step 4 cell with OVERWRITE_STEP4 = True for the full check.")

## What Step 5 picks up

The Step 4 outputs are **unsplit and unbalanced on purpose**:

1. `features_clean.csv` is the one Step 5 splits, then fits a scaler on the
   training rows only - using `features_scaled.csv` across a split would leak
   test statistics into training.
2. The class imbalance is untouched, and Step 5 leaves it untouched too. Check
   `labels.csv`: the smallest classes have very few rows, so a stratified split
   and macro-F1 / per-class recall matter far more than accuracy.
3. `metadata.csv` holds `Src IP`, `Dst IP`, the ports and the timestamps. They
   stay **out of the feature matrix** - attacker IPs decided the labels in
   Step 1, so training on them is label leakage - but the file is kept, because
   the sequence work needs it and because it carries each row back to its flow.


---

# Step 5 - Train-Test Validation Split

A single stratified 80/20 split of the Step 4 outputs, and **nothing else**.

| Step 5 does | Step 5 does not |
|---|---|
| a stratified split on row **positions**, so features, labels and metadata stay aligned | handle the class imbalance - no SMOTE, no resampling, no class weights |
| refit the scaler on the **training rows only** | select features |
| report how much of the test set is feature-identical to training | train any model |

Three settings decide everything, all in `config.py`:

```python
TEST_SIZE       = 0.20      # 20% of the rows go to test
RANDOM_STATE    = 42        # fixed, so the split is reproducible
STRATIFY_COLUMN = "Label"   # the MULTICLASS label
OVERWRITE_STEP5 = False     # keep existing outputs instead of resplitting
```

**Why stratify on `Label` and not `Label_binary`.** Stratifying on the binary
target would balance attack-vs-benign while letting a 23-row class land entirely
on one side of the split - and a class with no test rows cannot be evaluated at
all. Stratifying on the multiclass label keeps every attack type on both sides.

**Why the split is on indices.** `train_test_split` is given `np.arange(n_rows)`,
not the frames. The three frames are then selected with the same positions, so
row *i* of `X_train`, `y_train` and `meta_train` is guaranteed to be the same
flow. Splitting each frame separately would be three independent shuffles.


### Step 5 settings

In [ ]:
from src.mscad import splitting

splitting.show_table(splitting.kv_table([
    ("input folder", str(config.STEP4_OUTPUT_DIR)),
    ("output folder", str(config.STEP5_OUTPUT_DIR)),
    ("TEST_SIZE", config.TEST_SIZE),
    ("RANDOM_STATE", config.RANDOM_STATE),
    ("STRATIFY_COLUMN", config.STRATIFY_COLUMN),
    ("SCALER (refitted on train)", config.SCALER),
    ("CLIP_TEST_TO_TRAIN_RANGE", config.CLIP_TEST_TO_TRAIN_RANGE),
    ("OVERWRITE_STEP5", config.OVERWRITE_STEP5),
    ("outputs already exist", splitting.step5_outputs_exist()),
]), note="With `OVERWRITE_STEP5 = False` and the outputs already present, they "
         "are loaded instead of resplit, so the split never changes under you.")

### Run Step 5

`run_step5()` does 5.0 input checks, 5.1 the split, 5.2 the refitted scaler,
5.3 the leakage count, then writes every file and re-reads it from disk.


In [ ]:
step5 = splitting.run_step5()

## 5.1 Split - class balance

The table below is the one that matters. `share drift pp` is how far a class's
share of the training rows differs from its share of the test rows, in
percentage points. A stratified split can only move **whole rows**, so the
smallest achievable drift is one test row's worth of share; the check uses
whichever is larger, that figure or `config.SHARE_TOLERANCE_PP`.


In [ ]:
splitting.show_table(step5["balance"],
                     note="Per class: counts and shares in the full dataset, train "
                          "and test. Saved as class_balance.csv, so it is shown "
                          "whether Step 5 just ran or was loaded from disk.")

### Binary target (benign vs attack)

In [ ]:
splitting.show_table(step5["binary_balance"],
                     note="`Label_binary` is 0 for Normal/Benign and 1 for every "
                          "attack. It is NOT what the split was stratified on -- it "
                          "is shown to confirm the binary target came out balanced "
                          "as well.")

### Class counts, train vs test

Log scale, because the classes differ by four orders of magnitude. Only the test
bars carry a number: the question this chart answers is whether every class still
has test rows.


In [ ]:
chart = config.STEP5_OUTPUT_DIR / config.STEP5_FILES["chart"]
if chart.exists():
    if splitting.dl.in_notebook():
        from IPython.display import Image, display
        display(Image(filename=str(chart)))
    else:
        print("chart saved to", chart)
else:
    print("No chart at", chart)
    print("Install matplotlib and re-run Step 5 with OVERWRITE_STEP5 = True.")

## 5.2 Scale - fitted on the training rows only

Step 4's scaler saw every row, test rows included, so its min/max carry
information from the test set. Step 5 therefore starts from the **unscaled**
`features_clean.csv` and fits a fresh scaler on `X_train` alone.

Scaled test values outside `[0, 1]` are **expected**: the scaler never saw those
rows, so a test flow can be larger than every training flow. They are counted and
reported, not clipped - `CLIP_TEST_TO_TRAIN_RANGE = False`, because clipping
would quietly flatten real differences between large flows.

The one-hot `Protocol_*` columns and the `zero_duration` marker are left alone;
they are already 0/1.


In [ ]:
if "scale_info" in step5:
    info = step5["scale_info"]
    splitting.show_table(splitting.kv_table([
        ("scaler", info["scaler_name"]),
        ("fitted on", "X_train only (%s rows)" % f"{len(step5['X_train']):,}"),
        ("columns scaled", "%d of %d" % (info["n_columns"], step5["X_train"].shape[1])),
        ("columns left alone", ", ".join(
            c for c in step5["X_train"].columns if c not in set(info["columns"]))),
        ("train range", "[%.6f, %.6f]" % (info["train_min"], info["train_max"])),
        ("test range", "[%.6f, %.6f]" % (info["test_min"], info["test_max"])),
        ("test cells outside [0, 1]", f"{info.get('test_cells_outside_0_1', 0):,}"),
        ("clipped", "no" if not info["clipped"] else "yes"),
        ("saved scaler", str(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["scaler"])),
    ]))
    if len(info["out_of_range"]):
        splitting.show_table(info["out_of_range"].head(20),
                             note="Per column: how many test values fall outside "
                                  "[0, 1], and how far. Expected and small.")
else:
    # the loaded path: the same numbers, read back from split_summary.csv
    splitting.show_table(step5["split_summary"],
                         note="Read from split_summary.csv -- Step 5 was loaded from "
                              "disk, so there is no in-memory scaler report. Set "
                              "`OVERWRITE_STEP5 = True` to recompute it.")

## 5.3 Leakage check - reported, nothing dropped

About half the rows in this dataset share identical feature values once the
identifiers are removed: thousands of scan, flood and brute-force flows genuinely
look alike. A stratified random split therefore puts feature-identical twins on
both sides.

> **This inflates every test score and belongs in the results.** A model can
> "recognise" a test row it has effectively already seen in training, so the
> headline accuracy is optimistic. Report this percentage next to it, and lean on
> per-class recall for the classes whose share here is **low** - those are the
> rows the model actually had to generalise to.
>
> Nothing is dropped. These are real flows, and deleting them would change the
> dataset and break the attack-sequence work that comes later.


In [ ]:
splitting.show_table(step5["leakage_table"],
                     note="An exact match on the full UNSCALED feature vector. Saved "
                          "as leakage.csv.")

total = step5["leakage_table"].iloc[-1]
print("%s of %s test rows (%s%%) have a feature-identical twin in training."
      % (f"{int(total['also in train']):,}", f"{int(total['test rows']):,}",
         total["% of this class's test rows"]))

## 5.4 Saved files

Everything below is in `C:\MSCAD2022\outputs\step5\`.

| file | what it is |
|---|---|
| `X_train.csv` / `X_test.csv` | the features, scaled with the **train-fitted** scaler - this is what a model trains and is evaluated on |
| `X_train_unscaled.csv` / `X_test_unscaled.csv` | the same rows straight from `features_clean.csv`, for a model that wants its own scaling |
| `y_train.csv` / `y_test.csv` | `Label`, `Label_encoded`, `Label_binary` |
| `meta_train.csv` / `meta_test.csv` | the identifiers, including `Orig_Row`, so any row goes back to its flow |
| `train_indices.csv` / `test_indices.csv` | the row positions in the Step 4 files - the split, reproducible without rerunning it |
| `train_view.csv` / `test_view.csv` | the `preprocessing.csv` layout, **for viewing only** |
| `scaler_train.joblib` | the fitted scaler, with the settings it was fitted under |
| `split_summary.csv`, `step5_report.csv` | the numbers and the actions |
| `split_class_counts.png` | the chart above |

`X_train.csv` is the training matrix: the identifiers are not in it, and the
`Protocol_*` one-hot columns are. The two `*_view.csv` files are the readable
counterpart and must never be fed to a model.


In [ ]:
import pandas as pd

folder = config.usable_output_dir(config.STEP5_OUTPUT_DIR)
if folder is None:
    print("Step 5 outputs are not readable here:",
          config.why_unusable(config.STEP5_OUTPUT_DIR))
else:
    rows = []
    for key, name in config.STEP5_FILES.items():
        path = folder / name
        rows.append({"file": name, "exists": path.exists(),
                     "size (MB)": round(path.stat().st_size / 1024 / 1024, 3)
                                  if path.exists() else 0.0,
                     "what": key})
    splitting.show_table(pd.DataFrame(rows), note="Folder: %s" % folder)

### Shapes, and the head of each side

In [ ]:
X_train = pd.read_csv(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["X_train"],
                      low_memory=False)
X_test = pd.read_csv(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["X_test"],
                     low_memory=False)
y_train = pd.read_csv(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["y_train"])
y_test = pd.read_csv(config.STEP5_OUTPUT_DIR / config.STEP5_FILES["y_test"])

splitting.show_table(splitting.kv_table([
    ("X_train", "%s rows x %d columns" % (f"{X_train.shape[0]:,}", X_train.shape[1])),
    ("X_test", "%s rows x %d columns" % (f"{X_test.shape[0]:,}", X_test.shape[1])),
    ("y_train", "%s rows x %d columns" % (f"{y_train.shape[0]:,}", y_train.shape[1])),
    ("y_test", "%s rows x %d columns" % (f"{y_test.shape[0]:,}", y_test.shape[1])),
    ("train + test", f"{len(X_train) + len(X_test):,}"),
    ("identical columns", list(X_train.columns) == list(X_test.columns)),
]))

counts = y_test[config.LABEL_COLUMN].value_counts()
splitting.show_table(pd.DataFrame({"Label": counts.index.astype(str),
                                   "test rows": counts.to_numpy()}),
                     note="Per-class test counts -- every class must appear here.")
splitting.show_table(X_train.head(5), all_columns=True,
                     note="X_train.head(5), all %d columns." % X_train.shape[1])

### Verification

Every file above was re-read from disk and checked: the row counts add up to the
Step 4 total, no row position is in both sides, `X_train` and `X_test` have the
same columns in the same order, the labels and metadata line up with the split
positions (compared on `Orig_Row`), and neither matrix holds a NaN or an inf.


In [ ]:
if "saved" in step5 and "verification" in step5.get("saved", {}):
    splitting.show_table(step5["saved"]["verification"])
    print()
    print("ALL CHECKS PASSED:", step5["saved"]["ok"])
else:
    print("Step 5 was loaded from disk, so there is no fresh verification table.")
    print("Set OVERWRITE_STEP5 = True and re-run to split and verify again.")

## What Step 6 picks up

1. **Train on `X_train.csv` only.** `X_test.csv` is touched once, at the end, to
   report a score. Any choice made by looking at the test set - a threshold, a
   feature, a hyper-parameter - makes that score meaningless.
2. **The imbalance is still there, on purpose.** SMOTE, resampling or class
   weights belong in Step 6 and must be fitted on the **training** side of this
   split only. Oversampling before the split copies rows across it, which is the
   same leak as scaling before the split.
3. **Use `train_indices.csv` / `test_indices.csv`** to carry this exact split
   into any other experiment, instead of resplitting with the same seed and
   hoping.
4. **Report the 5.3 leakage percentage next to every score.** Macro-F1 and
   per-class recall, not accuracy: with Brute_Force at tens of thousands of rows
   and Web_Crwling at 23, accuracy is nearly a measure of the majority class.


---

# Step 6 - Feature Selection (MI + Pearson + PI -> top 30)

66+ features in, 30 out. Three measures, each answering a different question:

| | what it measures | blind to |
|---|---|---|
| **6.2 Mutual information** | how much knowing the feature tells you about the class | every other feature - two identical features both score highly |
| **6.3 Pearson \|r\|** | how much two **features** repeat each other. No labels at all | whether either is useful |
| **6.4 Permutation importance** | how far the forest's macro-F1 falls when that one column is shuffled | a feature whose partner can stand in for it |

**Why all three, and why MI and PI are added rather than one being picked.**
They fail in opposite directions. PI understates a feature whose partner is
still in the set at \|r\| around 0.80-0.95: shuffle one and the forest simply
leans on the other, so neither looks important. MI never sees other features, so
that cannot happen to it. Conversely MI rewards a feature that merely tracks the
label without adding anything the others do not already give; PI catches those.
Adding the two normalised scores means a feature has to fail *both* tests to be
dropped.

**Everything is decided on the training set.** MI, the correlation matrix, the
inner fit/validation split and the permutation importance all run on `X_train`
alone. `X_test` is touched once, in 6.6, to take the same 30 columns in the same
order. Choosing features by looking at the test set is the same leak as scaling
before the split.

> **Step 6 does nothing else.** No autoencoder, no CNN-LSTM, no modelling. No
> class-imbalance handling - no SMOTE, no resampling. The RandomForest in 6.4
> exists only to rank features; 6.4 explains why its score is not a result.


### Step 6 settings

In [ ]:
from src.mscad import feature_selection

feature_selection.show_table(feature_selection.kv_table([
    ("input folder", str(config.STEP5_OUTPUT_DIR)),
    ("output folder", str(config.STEP6_OUTPUT_DIR)),
    ("PEARSON_THRESHOLD", config.PEARSON_THRESHOLD),
    ("PEARSON_REPORT_LOWER", config.PEARSON_REPORT_LOWER),
    ("N_SELECTED_FEATURES", config.N_SELECTED_FEATURES),
    ("RF_PARAMS", str(config.RF_PARAMS)),
    ("PI_SCORING / PI_N_REPEATS", "%s / %d" % (config.PI_SCORING, config.PI_N_REPEATS)),
    ("PI_VALIDATION_SIZE", config.PI_VALIDATION_SIZE),
    ("MI_N_NEIGHBORS", config.MI_N_NEIGHBORS),
    ("CLIP_NEGATIVE_PI", config.CLIP_NEGATIVE_PI),
    ("OVERWRITE_STEP6", config.OVERWRITE_STEP6),
    ("outputs already exist", feature_selection.step6_outputs_exist()),
]), note="`PI_SCORING = \"f1_macro\"` weights every class equally, so a feature "
         "that only the 23-row classes depend on still registers. With `accuracy` "
         "the 84% Brute_Force class would drown them out entirely.")

### Run Step 6

**This cell is slow.** Permutation importance refits nothing but re-scores the
validation rows `n_repeats` times per feature - on 66 features at 10 repeats
that is 660 scorings, and it can take several minutes. The elapsed time for MI
and for PI is printed as each finishes.

With `OVERWRITE_STEP6 = False` and the outputs already present, they are loaded
from disk instead, which is why that setting is worth keeping.


In [ ]:
step6 = feature_selection.run_step6()

## 6.1 Training data overview

The input is `X_train.csv` / `X_test.csv` from Step 5, already scaled with the
train-fitted scaler. Before anything is scored, four things are confirmed: the
two frames carry identical columns in identical order, the labels line up, no
label column has crept into the features, and no identifier has either.

A column that is **constant** across the training rows is dropped here. Pearson
correlation is undefined for it (zero variance means dividing by zero), so it
would otherwise put a row and a column of `NaN` into the matrix and quietly
corrupt the redundancy filter. A constant also tells a model nothing, so nothing
is lost.


In [ ]:
if step6.get("status") != "loaded_existing":
    feature_selection.show_table(step6["classes"])
    dropped = step6["constant_dropped"]
    print("constant columns dropped:", ", ".join(dropped) if dropped else "(none)")
    print("features after the drop  :", len(step6["mi"]))
else:
    print("Step 6 was loaded from disk, so the in-memory tables are not available.")
    print("Set OVERWRITE_STEP6 = True in config.py and re-run the cell above.")

## 6.2 Mutual information

`mutual_info_classif` between every feature and `Label_encoded`, on the training
rows. MI is a *statistical dependency* measure: unlike a correlation it finds
non-linear relationships, and it makes no assumption that the relationship has a
direction.

Two details that matter:

- The one-hot and binary columns (`Protocol_*`, `zero_duration`) are declared
  **discrete**. Left as continuous, the k-nearest-neighbour estimator would
  treat a column of 0s and 1s as a dense number line and return a wrong score.
- MI is **stochastic** - it jitters continuous inputs before estimating - so it
  is computed **once** per run and the same numbers are reused for the Pearson
  tie-break in 6.3 and the ranking in 6.5. Recomputing it would give slightly
  different values and the two stages would silently disagree.


In [ ]:
if step6.get("status") != "loaded_existing":
    feature_selection.show_table(step6["mi_table_all"].head(25),
                                 note="Top 25 of %d by mutual information. Saved in "
                                      "full as mi_scores_all.csv."
                                      % len(step6["mi_table_all"]))
    print("MI computed in %.1f s" % step6["mi_seconds"])
else:
    feature_selection.show_table(step6["mi_all"].head(25),
                                 note="Read back from mi_scores_all.csv.")

## 6.3 Pearson redundancy filter

The absolute correlation between every **pair of features**. No labels are
involved: this step is only about features repeating each other.

For every pair at `|r| >= 0.95` one of the two is dropped, keeping whichever has
the higher MI from 6.2. The walk is deterministic by construction: pairs are
processed from the highest `|r|` downward, a pair is skipped once either side has
already gone, and an MI tie falls back to feature order - so the result never
depends on dictionary or set iteration order.

CICFlowMeter produces several near-copies by design - `Subflow Fwd Pkts` is
usually `Tot Fwd Pkts`, `Pkt Size Avg` tracks `Pkt Len Mean` - which is what this
filter is for. Expect the `Protocol_*` one-hot columns to show up here too: with
three protocols, two of the indicators determine the third, so they are genuinely
redundant.


In [ ]:
if step6.get("status") != "loaded_existing":
    dropped = step6["dropped_pairs"]
    if len(dropped):
        feature_selection.show_table(dropped,
            note="`MI kept` is always >= `MI dropped` - the higher-MI half of each "
                 "pair survives. Saved as pearson_dropped_pairs.csv.")
    else:
        print("No pair reached |r| >= %.2f, so nothing was dropped."
              % config.PEARSON_THRESHOLD)
    print("features: %d  ->  %d" % (len(step6["mi"]), len(step6["pearson_kept"])))

### Still correlated, but kept

These pairs sit in `[0.80, 0.95)`. They are not redundant enough to drop, but
they are exactly the pairs permutation importance will **understate**: shuffle
one and the forest leans on the other, so neither looks important. That is the
concrete reason MI is added to PI rather than replaced by it.


In [ ]:
if step6.get("status") != "loaded_existing":
    remaining = step6["remaining_pairs"]
    if len(remaining):
        feature_selection.show_table(remaining.head(25),
            note="%d pair(s) in [%.2f, %.2f). Saved as pearson_remaining_pairs.csv."
                 % (len(remaining), config.PEARSON_REPORT_LOWER,
                    config.PEARSON_THRESHOLD))
    else:
        print("No remaining pair in [%.2f, %.2f)."
              % (config.PEARSON_REPORT_LOWER, config.PEARSON_THRESHOLD))

### Correlation heatmap

One hue, light to dark: `|r|` is a magnitude, so a single blue ramp reads it
correctly. The dark diagonal is each feature against itself; any other dark
square is a redundant pair.


In [ ]:
heatmap = config.STEP6_OUTPUT_DIR / config.STEP6_FILES["heatmap"]
if heatmap.exists():
    if feature_selection.dl.in_notebook():
        from IPython.display import Image, display
        display(Image(filename=str(heatmap)))
    else:
        print("heatmap saved to", heatmap)
else:
    print("No heatmap at", heatmap)
    print("Install matplotlib and re-run Step 6 with OVERWRITE_STEP6 = True.")

## 6.4 Permutation importance

The training set is split **again**, into an inner fit part and an inner
validation part (stratified, 80/20). A RandomForest is fitted on the fit part;
then each feature's column in the validation part is shuffled and the drop in
macro-F1 is measured. A feature the forest relies on loses it a lot of score; a
feature it ignores loses it nothing.

> **The baseline macro-F1 below is a ranking tool, not a model result.** Around
> half of this dataset's rows share identical feature values once the identifiers
> are gone - thousands of scan, flood and brute-force flows genuinely look alike -
> so feature-identical rows sit in both the fit and the validation part. The
> forest is scored partly on rows it has already seen. The percentage is printed
> below; quote Step 7's numbers, never this one.

> **PI understates correlated features.** For any pair still at `|r|` 0.80-0.95
> after 6.3, shuffling one leaves the other intact and the forest simply uses it,
> so both score low. MI is immune to this, which is the whole reason 6.5 adds the
> two instead of trusting either alone.


In [ ]:
if step6.get("status") != "loaded_existing":
    info = step6["pi_info"]
    feature_selection.show_table(feature_selection.kv_table([
        ("inner fit rows", f"{info['fit_rows']:,}"),
        ("inner validation rows", f"{info['validation_rows']:,}"),
        ("forest fit time", "%.1f s" % info["fit_seconds"]),
        ("permutation time", "%.1f s" % info["pi_seconds"]),
        ("scoring", "%s, %d repeats" % (config.PI_SCORING, config.PI_N_REPEATS)),
        ("baseline macro-F1", "%.4f" % info["baseline_macro_f1"]),
        ("validation rows also in the fit part", "%s (%.2f%%)"
         % (f"{info['overlap']['also in fit']:,}", info["overlap"]["% of validation"])),
    ]))
    feature_selection.show_table(info["per_class_f1"],
                                 note="Per-class F1 on the inner validation part.")
    if info["classes_missing_from_validation"]:
        print("[WARN] no validation rows for:",
              ", ".join(info["classes_missing_from_validation"]))
    if info["thin_validation_classes"]:
        print("[WARN] fewer than 2 validation rows for:",
              info["thin_validation_classes"])
    feature_selection.show_table(step6["pi_table"].head(25),
                                 note="Top 25 by permutation importance. Saved in "
                                      "full as pi_scores.csv.")
else:
    feature_selection.show_table(step6["pi_scores"].head(25),
                                 note="Read back from pi_scores.csv.")

### How much of the inner validation part is already in the fit part

In [ ]:
if step6.get("status") != "loaded_existing":
    feature_selection.show_table(step6["overlap_table"],
        note="Exact matches on the full feature vector. The classes with a LOW "
             "percentage here are the ones the forest actually had to generalise "
             "to - weigh their F1 accordingly.")
else:
    feature_selection.show_table(step6["pi_overlap"],
                                 note="Read back from pi_duplicate_overlap.csv.")

## 6.5 Combine MI and PI

The two scores live on completely different scales - MI is in nats, PI is a
difference in F1 - so adding them raw would let whichever happens to be larger
decide everything. Both are min-max normalised across the surviving features
first:

$$\text{MI}' = \frac{\text{MI} - \min}{\max - \min}, \qquad
  \text{PI}' = \frac{\text{PI} - \min}{\max - \min}, \qquad
  \text{CombinedScore} = \text{MI}' + \text{PI}'$$

Each part contributes at most 1.0, so the combined score runs 0 to 2 and neither
measure can dominate by living on a bigger scale. Two edge cases are handled
explicitly rather than left to produce `NaN`: a **negative** PI (shuffling made
the score *better*, which is noise around zero, not evidence against the feature)
is clipped to 0 so it cannot stretch the scale, and a **flat** score where
`max == min` normalises to 0 for every feature with a warning, leaving the
ranking to the other measure.


In [ ]:
ranking = step6["ranking"]
feature_selection.show_table(ranking.head(35),
    note="Sorted by CombinedScore, ties broken by MI' and then by feature name, so "
         "the order is fully determined. Saved in full as combined_ranking.csv.")

## 6.6 The selected features

The top `N_SELECTED_FEATURES` by combined score. If the Pearson filter left fewer
than that, all of them are selected and a warning says so.


In [ ]:
selected = step6["selected"]
chosen = ranking[ranking["selected"]] if "selected" in ranking.columns \
    else ranking.head(len(selected))
feature_selection.show_table(
    chosen[["rank", "feature", "MI", "PI", "MI_norm", "PI_norm", "CombinedScore"]],
    note="The %d selected features, in rank order." % len(selected))

cut = float(chosen["CombinedScore"].iloc[-1])
rest = ranking[~ranking["feature"].isin(set(selected))]
if len(rest):
    print("lowest selected : %.4f  (%s)" % (cut, chosen["feature"].iloc[-1]))
    print("highest rejected: %.4f  (%s)"
          % (float(rest["CombinedScore"].iloc[0]), rest["feature"].iloc[0]))
    print("gap at the cut  : %.4f" % (cut - float(rest["CombinedScore"].iloc[0])))

### Top 30 by combined score

Each bar is split into its two parts, so a feature carried by MI alone and one
carried by PI alone are visibly different even though they land at the same
total. A bar that is almost entirely orange is a feature the forest leans on
that MI barely notices; an almost entirely blue one is the reverse - and that one
is often half of a correlated pair whose PI was suppressed.


In [ ]:
chart = config.STEP6_OUTPUT_DIR / config.STEP6_FILES["ranking_chart"]
funnel = config.STEP6_OUTPUT_DIR / config.STEP6_FILES["funnel"]
if feature_selection.dl.in_notebook():
    from IPython.display import Image, display
    for picture in (chart, funnel):
        if picture.exists():
            display(Image(filename=str(picture)))
        else:
            print("missing:", picture)
else:
    print("charts:", chart, funnel)

### Saved files

Everything below is in `C:\MSCAD2022\outputs\step6\`.

| file | what it is |
|---|---|
| `X_train_selected.csv` / `X_test_selected.csv` | the 30 selected columns, same order in both - **this is what Step 7 trains on** |
| `selected_features.json` | the ordered names plus every setting they were chosen under |
| `combined_ranking.csv` | every surviving feature with MI, PI, both normalised scores and `selected` |
| `mi_scores_all.csv` / `mi_scores.csv` | MI before and after the Pearson filter |
| `pearson_corr_matrix.csv`, `pearson_dropped_pairs.csv`, `pearson_remaining_pairs.csv` | the redundancy filter's working |
| `pi_scores.csv`, `rf_per_class_f1.csv`, `pi_duplicate_overlap.csv` | the permutation stage |
| `y_train.csv` / `y_test.csv` | copied from Step 5, unchanged |
| `rf_baseline.joblib` | the ranking forest - **not** a model for reporting |
| `step6_report.csv` | feature counts at each stage, and what was done |


In [ ]:
import pandas as pd

folder = config.usable_output_dir(config.STEP6_OUTPUT_DIR)
if folder is None:
    print("Step 6 outputs are not readable here:",
          config.why_unusable(config.STEP6_OUTPUT_DIR))
else:
    rows = []
    for key, name in config.STEP6_FILES.items():
        path = folder / name
        rows.append({"file": name, "exists": path.exists(),
                     "size (MB)": round(path.stat().st_size / 1024 / 1024, 3)
                                  if path.exists() else 0.0,
                     "what": key})
    feature_selection.show_table(pd.DataFrame(rows), note="Folder: %s" % folder)

    X_train_sel = pd.read_csv(folder / config.STEP6_FILES["X_train_selected"],
                              low_memory=False)
    X_test_sel = pd.read_csv(folder / config.STEP6_FILES["X_test_selected"],
                             low_memory=False)
    feature_selection.show_table(feature_selection.kv_table([
        ("X_train_selected", "%s rows x %d columns"
         % (f"{X_train_sel.shape[0]:,}", X_train_sel.shape[1])),
        ("X_test_selected", "%s rows x %d columns"
         % (f"{X_test_sel.shape[0]:,}", X_test_sel.shape[1])),
        ("identical columns, in order",
         list(X_train_sel.columns) == list(X_test_sel.columns)),
    ]))
    feature_selection.show_table(X_train_sel.head(5), all_columns=True,
                                 note="X_train_selected.head(5).")

### Verification

Every file was re-read from disk and checked: exactly N columns in the same order
in train and test, row counts still matching Step 5, labels aligned, no label or
identifier among the selected columns, no NaN or inf, and
`CombinedScore == MI_norm + PI_norm` in the saved ranking.


In [ ]:
if "saved" in step6 and "verification" in step6.get("saved", {}):
    feature_selection.show_table(step6["saved"]["verification"])
    print()
    print("ALL CHECKS PASSED:", step6["saved"]["ok"])
else:
    print("Step 6 was loaded from disk, so there is no fresh verification table.")
    print("Set OVERWRITE_STEP6 = True and re-run to select and verify again.")

## What Step 7 picks up

1. **Train on `X_train_selected.csv`.** `X_test_selected.csv` is touched once, at
   the end, to report a score. It already holds the same 30 columns in the same
   order, so no further alignment is needed.
2. **The imbalance is still untouched.** SMOTE, resampling or class weights go in
   Step 7 and must be fitted on the **training** side only. Oversampling before
   the split copies rows across it - the same leak as scaling before the split.
3. **The 30 features were chosen on the training rows alone**, so they carry no
   information from the test set. Re-running selection on the full dataset would
   quietly undo that.
4. **Report the Step 5.3 duplicate percentage next to every score**, and prefer
   macro-F1 and per-class recall over accuracy: with Brute_Force at tens of
   thousands of rows and Web_Crwling at 23, accuracy is close to a measure of the
   majority class alone.


---

# Step 7 - Autoencoder feature extraction

An autoencoder is a network trained to rebuild **its own input** through a
narrow middle layer. Squeeze 30 features through 15 numbers and force the
network to reconstruct the 30, and the 15 have to carry whatever structure the
30 share. Those 15 are the new feature set for Steps 10-12.

```
    30 features  ->  24  ->  15 (latent)  ->  24  ->  30 reconstruction
                 \___ encoder ___/        \___ decoder ___/
```

Only the **encoder** half is kept for later use. The decoder exists to create
the training signal and is then discarded.

**It is unsupervised.** The loss is the reconstruction error against the input
itself, so no label is ever a target. Labels appear in Step 7 for exactly one
purpose: to *stratify* the inner split used for early stopping, so the 23-row
classes are represented in the validation part instead of landing entirely in
the fit part by chance. Nothing about the encoding depends on them.

**Why a sigmoid output.** Step 5's MinMax scaler put every training feature into
`[0, 1]`, so a sigmoid can reach every legal value and nothing else. A linear
output would be free to predict -3 or 7 - values no input can take - and would
waste capacity learning not to.

> **Framework: TensorFlow/Keras, not PyTorch.** The rule was "PyTorch unless
> TensorFlow/Keras officially supports Windows + Python 3.13", and it does:
> TensorFlow 2.21.0 publishes `tensorflow-2.21.0-cp313-cp313-win_amd64.whl` on
> PyPI, so `pip install tensorflow` works on this machine with no source build.
> That wheel is CPU-only - native-Windows GPU support ended after TF 2.10 -
> which is all these steps need. The same framework is used for Steps 10-12, so
> there is one set of seeds, one save format and one set of versions to cite.
>
> ```
> "C:\Users\banot\AppData\Local\Programs\Python\Python313\python.exe" -m pip install "tensorflow>=2.21,<3"
> ```

**Reproducibility.** `set_all_seeds()` seeds Python, NumPy and Keras from
`RANDOM_STATE` and calls `tf.config.experimental.enable_op_determinism()`.
Without that last call the same seed still drifts run to run, because several
CPU kernels reduce in a non-deterministic order. With it, two runs give
bit-identical codes - there is a test for exactly that.


### Step 7 settings

In [ ]:
from src.mscad import autoencoder

versions = autoencoder.framework_versions()
autoencoder.show_table(autoencoder.kv_table([
    ("input folder", str(config.STEP6_OUTPUT_DIR)),
    ("Step 7 output", str(config.STEP7_OUTPUT_DIR)),
    ("Step 8 output", str(config.STEP8_OUTPUT_DIR)),
    ("architecture", "%d -> %s -> %d -> %s -> %d"
     % (config.N_SELECTED_FEATURES, config.AE_HIDDEN, config.AE_LATENT_DIM,
        config.AE_HIDDEN, config.N_SELECTED_FEATURES)),
    ("activation / output", "%s / %s" % (config.AE_ACTIVATION, config.AE_OUTPUT_ACT)),
    ("loss", config.AE_LOSS),
    ("epochs / batch / lr", "%d / %d / %s"
     % (config.AE_EPOCHS, config.AE_BATCH_SIZE, config.AE_LR)),
    ("early stopping", "val_size %s, patience %d, best weights restored"
     % (config.AE_VAL_SIZE, config.AE_PATIENCE)),
    ("framework", "%s %s, keras %s"
     % (versions["framework"], versions["tensorflow"], versions["keras"])),
    ("python / numpy", "%s / %s" % (versions["python"], versions["numpy"])),
    ("OVERWRITE_STEP7 / STEP8", "%s / %s"
     % (config.OVERWRITE_STEP7, config.OVERWRITE_STEP8)),
    ("outputs already exist", "%s / %s"
     % (autoencoder.step7_outputs_exist(), autoencoder.step8_outputs_exist())),
]), note="These versions are written into ae_config.json, so a result can be "
         "reproduced from the file alone.")

### Run Step 7

Training is the slow part. On a CPU with ~84,000 training rows, 100 epochs at
batch 256 takes a few minutes; early stopping usually ends it sooner. The
elapsed time and the epoch it stopped at are both reported.


In [ ]:
step7 = autoencoder.run_step7()

## 7.1 Architecture

`units` is the width of each layer and `parameters` is how many weights it
holds. The decoder mirrors the encoder's **shape**, not its values - the two
halves have separate weights, which is why the parameter count is roughly
symmetric rather than halved.


In [ ]:
if step7.get("status") != "loaded_existing":
    autoencoder.show_table(step7["architecture"],
        note="Only the encoder half (input through `latent`) is used in Step 8.")
else:
    autoencoder.show_table(step7["architecture"],
                           note="Read back from architecture.csv.")

## 7.2 Training

The numbers below are **reconstruction errors**, not classification scores: the
target is the input itself. A validation MSE close to the training MSE means the
bottleneck is genuinely compressing rather than memorising.


In [ ]:
info = step7["info"]
autoencoder.show_table(autoencoder.kv_table([
    ("best epoch", info.get("best_epoch")),
    ("epochs run", info.get("epochs_run")),
    ("train reconstruction MSE", "%.6f" % info["train_mse"]),
    ("validation reconstruction MSE", "%.6f" % info["val_mse"]),
    ("training time", "%.1f s" % info["seconds"] if info.get("seconds") else "-"),
]), note="Early stopping restored the weights from the best epoch, so these are "
         "the weights that were saved - not the last epoch's.")

curve = config.STEP7_OUTPUT_DIR / config.STEP7_FILES["loss_curve"]
if curve.exists() and autoencoder.dl.in_notebook():
    from IPython.display import Image, display
    display(Image(filename=str(curve)))
elif not curve.exists():
    print("No loss curve at", curve)

### Per-feature reconstruction error

Which of the 30 features the 15-number bottleneck could **not** keep. A high
value means that feature is either noisy or carries something none of the others
do - worth knowing before Steps 10-12, because whatever the autoencoder threw
away is not available to any classifier downstream.


In [ ]:
table = step7.get("per_feature_mse")
if table is not None:
    autoencoder.show_table(table,
        note="On the inner validation part. Saved as per_feature_mse.csv.")
else:
    print("Not available on the loaded path - set OVERWRITE_STEP7 = True to recompute.")

---

# Step 8 - Transform the training and test sets

The **same frozen encoder** runs over both sets:

```
    Z_train = encoder(X_train_selected)      Z_test = encoder(X_test_selected)
```

`X_test_selected` is read here for the first time in Steps 7-8, and only to be
pushed through an encoder that is already trained and unchanged. Nothing about
the architecture, the epochs or the early stopping came from it.

**Row order is unchanged**, so row *i* of `Z_train` is still row *i* of
`X_train_selected`, and `y_train` still belongs to it.

> **`encode()` checks the columns before every call.** A `Dense` layer reads
> **positions**, not names - hand it the same 30 columns in a different order and
> it returns numbers that look perfectly reasonable and are wrong. The saved
> encoder therefore ships with the feature order it was trained on, and `encode()`
> refuses any frame that does not match it exactly, naming the first position that
> differs.


In [ ]:
step8 = autoencoder.run_step8(
    encoder=step7.get("encoder"),
    autoencoder=step7.get("autoencoder"),
    feature_order=step7.get("feature_order"))

### The encoded matrices

In [ ]:
import pandas as pd

Z_train = step8["Z_train"]
Z_test = step8["Z_test"]
autoencoder.show_table(autoencoder.kv_table([
    ("Z_train", "%s rows x %d columns" % (f"{Z_train.shape[0]:,}", Z_train.shape[1])),
    ("Z_test", "%s rows x %d columns" % (f"{Z_test.shape[0]:,}", Z_test.shape[1])),
    ("columns", "%s ... %s" % (Z_train.columns[0], Z_train.columns[-1])),
    ("compression", "%d features -> %d numbers"
     % (config.N_SELECTED_FEATURES, config.AE_LATENT_DIM)),
]))
autoencoder.show_table(Z_train.head(5), all_columns=True,
                       note="Z_train.head(5). These are the latent codes, not the "
                            "original features - they have no individual meaning.")

if "recon_mse" in step8:
    autoencoder.show_table(step8["recon_mse"],
        note="**Reported only.** No decision in Steps 7-8 was made by looking at "
             "the test error. A test MSE close to the train MSE means the encoder "
             "generalises; a much larger one would mean it had memorised the "
             "training rows.")

### Verification

Re-read from disk: `Z_train` is `(N_train, 15)`, `Z_test` is `(N_test, 15)`, both
carry `z01 ... z15` in the same order, no NaN or inf, and the labels are still
aligned row for row.


In [ ]:
if "verification" in step8:
    autoencoder.show_table(step8["verification"])
    print()
    print("ALL CHECKS PASSED:", step8["ok"])
else:
    print("Step 8 was loaded from disk, so there is no fresh verification table.")
    print("Set OVERWRITE_STEP8 = True and re-run to encode and verify again.")

### Re-using the encoder later

The encoder is saved as a Keras model plus the feature order it expects, so any
later step can load it without touching Step 7:

```python
from src.mscad import autoencoder
encode, model, feature_order = autoencoder.load_encoder()
Z = encode(X)          # raises unless X has exactly those columns, in that order
```


## What Step 9 picks up

1. **Train on `Z_train.csv`** (15 columns) with `y_train.csv`. `Z_test.csv` is
   touched once, at the end, to report a score.
2. **The 15 latent columns have no individual meaning.** `z07` is not a feature
   anyone can name - it is one axis of a learned compression. Feature-importance
   claims about `z` columns do not transfer back to the original 30; use the
   Step 6 ranking for that.
3. **The imbalance is still untouched.** It has been left alone deliberately
   through Steps 5-8. SMOTE, resampling or class weights belong in a later step
   and must be fitted on the **training** side only.
4. **Whatever the autoencoder discarded is gone** for every classifier
   downstream. The per-feature reconstruction table in 7.2 is the record of what
   that was.


---
# Stage 3 - preprocessing: clean, encode, scale

_(no code yet)_

---
# Stage 4 - feature selection (Mutual Information / Permutation Importance)

_(no code yet)_

---
# Stage 5 - autoencoder feature extraction

_(no code yet)_

---
# Stage 6 - class-imbalance handling

_(no code yet)_

---
# Stage 7 - models: RF, XGBoost, LSTM, CNN, hybrid CNN-LSTM

_(no code yet)_

---
# Stage 8 - evaluation, sliding windows, PrefixSpan, attack-state graph

This is why Step 1 and Step 2 keep Src IP / Dst IP / ports / protocol / Timestamp.

_(no code yet)_